# Lezioni UniVR verso Appunti e Schemi

App tutto-in-uno. Non serve saper programmare: esegui le celle in ordine (tasto play a sinistra di ogni cella) e compila i campi che compaiono.

Cosa fa: prendi una lezione (da Panopto UniVR, dal tuo PC, o da Google Drive), la trascrive, genera appunti / esercizi / schemi con diagrammi e immagini delle slide, te li mostra e li scarica.

Nota: servono due chiavi gratuite (Groq e Gemini) perche' la trascrizione e la scrittura degli appunti usano modelli di intelligenza artificiale online. Sono gratuite e si creano in un minuto (link nella cella 2).


## Cella 1 - Preparazione (esegui e aspetta circa 2 minuti)

In [ ]:
#@title Installa i componenti necessari
import os, subprocess
print("Installazione in corso, attendi...")
for c in [
    "apt-get -qq install -y ffmpeg mkvtoolnix",
    "pip install -q groq video-reader-rs",
    "git clone -q https://github.com/Slydite/lore-engine.git /content/lore-engine",
    "pip install -q -e /content/lore-engine",
    "git clone -q https://gitlab.com/Microeinstein/panopto-sync.git /content/panopto-sync",
    "pip install -q pycryptodomex urllib3 requests progress colorama greenlet ffmpeg-python yt-dlp",
]:
    subprocess.run(c, shell=True, capture_output=True)
os.makedirs("/content/lavoro", exist_ok=True)
print("Fatto. Vai alla cella 2.")


## Cella 2 - Chiavi API gratuite

Crea le due chiavi (gratis) e incollale nei campi, poi esegui la cella:
- Groq (trascrive l'audio): https://console.groq.com/keys
- Gemini (scrive gli appunti): https://aistudio.google.com/apikey

Le chiavi restano solo in questa sessione del browser, non vengono salvate altrove.

Nota: in teoria si potrebbe fare tutto in locale senza API (Whisper offline per la trascrizione, un modello di linguaggio locale per gli appunti), ma servirebbe una GPU, sarebbe piu' lento e la qualita' sarebbe inferiore a Gemini. Questa versione usa le API perche' sono gratuite, veloci e danno risultati migliori. Se un giorno servisse la versione 100% offline, si costruisce a parte.


In [ ]:
#@title Inserisci le chiavi e premi play
#@markdown Crea la chiave Groq qui: https://console.groq.com/keys
#@markdown (puoi metterne piu' di una separate da virgola: se una si esaurisce, passa alla successiva)
GROQ_KEY = ""   #@param {type:"string"}
#@markdown Crea la chiave Gemini qui: https://aistudio.google.com/apikey
GEMINI_KEY = "" #@param {type:"string"}

GROQ_KEY = GROQ_KEY.strip()
GEMINI_KEY = GEMINI_KEY.strip()

if not GROQ_KEY or not GEMINI_KEY:
    print("Manca almeno una chiave. Compila entrambi i campi e riesegui.")
else:
    with open("/content/lore-engine/.env", "w") as f:
        f.write(f"GEMINI_API_KEY_1={GEMINI_KEY}\n")
    print("Chiavi salvate per questa sessione. Vai alla cella 3.")


## Cella 3 - Da dove prendo la lezione?

Scegli una sola sorgente e poi esegui SOLO il gruppo di celle relativo:
- Panopto UniVR: celle 4, 5, 6 (login, cerca, scarica)
- Video dal PC: cella 7
- Video da Google Drive: cella 8

Le altre le salti.


## Cella 3bis - Usare Google Drive come memoria (consigliato)

Con Drive attivo, restano salvati sul tuo Drive: l'accesso a UniVR, l'elenco dei corsi e i video gia' scaricati. Cosi' la prossima volta riparti subito, senza rifare login ne' riscaricare le lezioni gia' prese. Senza Drive, chiudendo Colab perdi tutto e ricominci da capo ogni volta.


In [ ]:
#@title Usare Google Drive come memoria?
SALVA_SU_DRIVE = "si" #@param ["si", "no"]

import os
if SALVA_SU_DRIVE == "si":
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/LezioniUniVR"
    print("Memoria attiva su Drive:", BASE)
    print("Le lezioni gia' prese non verranno riscaricate.")
else:
    BASE = "/content/panopto-sync"
    print("Nessuna memoria: chiudendo Colab perderai le lezioni scaricate.")

LESSONS_DIR = BASE + "/lessons"
ID_FILE = BASE + "/default.id"
os.makedirs(LESSONS_DIR, exist_ok=True)
print("Vai alla cella 4.")


## Cella 4 - Panopto: accedi e scarica l'elenco delle lezioni

In [ ]:
#@title Credenziali GIA (le stesse del portale UniVR)
USERNAME_GIA = "" #@param {type:"string"}
PASSWORD_GIA = "" #@param {type:"string"}
#@markdown Esempio username: `ID732456` (il tuo identificativo GIA)
AGGIORNA_ELENCO = False #@param {type:"boolean"}
#@markdown Lascialo com'e' al primo accesso. Spuntalo solo se hai gia' usato l'app prima (con Drive) e vuoi ricontrollare se sono uscite lezioni nuove.

import os, sys, builtins, json, io, contextlib
sys.path.insert(0, "/content/panopto-sync")
os.chdir("/content/panopto-sync")
import input as input_module
from pathlib import Path

# se non hai eseguito la cella 3bis (memoria Drive), uso una memoria temporanea
if 'LESSONS_DIR' not in globals():
    BASE = "/content/panopto-sync"
    LESSONS_DIR = BASE + "/lessons"
    ID_FILE = BASE + "/default.id"
    os.makedirs(LESSONS_DIR, exist_ok=True)
    print("Nota: cella 3bis non eseguita. Uso memoria temporanea (nessun salvataggio su Drive).")
    print("Per non riscaricare ogni volta, torna alla cella 3bis ed eseguila.\n")

ELENCO = []
_sync_json = os.path.join(LESSONS_DIR, "sync.json")

def _carica_elenco():
    global ELENCO
    data = json.load(open(_sync_json, encoding="utf-8"))
    ELENCO = []
    for cid, corso in data["courses"].items():
        for uuid, lez in corso.get("lessons", {}).items():
            dur = lez.get("duration")
            mm = f" [{round(dur/1000/60)} min]" if dur else ""
            fname = lez.get("filename", lez["name"] + ".mkv")
            ELENCO.append((f"{corso['name']} - {lez['name']}{mm}", uuid, fname))
    ELENCO.sort()

# 1) credenziali: se gia' salvate (Drive) non richiede il login
if not os.path.exists(ID_FILE):
    if not (USERNAME_GIA and PASSWORD_GIA):
        print("Primo accesso: inserisci username e password GIA, poi riesegui.")
    else:
        print("Accesso in corso...")
        try:
            _oi, _og = builtins.input, input_module.getpass
            builtins.input = lambda p="": USERNAME_GIA
            _q = iter([PASSWORD_GIA, PASSWORD_GIA, "colabkey", "colabkey"])
            input_module.getpass = lambda p="": next(_q)
            try:
                input_module.Credentials.generate(Path(ID_FILE))
            finally:
                builtins.input, input_module.getpass = _oi, _og
            print("Credenziali salvate.")
        except Exception as e:
            print("Errore creazione credenziali:", e)
else:
    print("Credenziali gia' salvate, login automatico.")

# 2) elenco: usa quello salvato, oppure lo (ri)scarica
if os.path.exists(ID_FILE):
    if os.path.exists(_sync_json) and not AGGIORNA_ELENCO:
        _carica_elenco()
        print(f"Elenco riletto dalla memoria: {len(ELENCO)} lezioni. Vai alla cella 5.")
    else:
        print("Indicizzo i corsi (NON scarico ancora video). Puo' richiedere uno o due minuti...")
        try:
            _og2 = input_module.getpass
            input_module.getpass = lambda p="": "colabkey"
            _buf = io.StringIO()
            try:
                sys.argv = ["panoptoSync.py", "--idfile", ID_FILE, "sync", "--syncdir", LESSONS_DIR, "--only-meta"]
                import panoptoSync
                panoptoSync.check_missing_download_softwares = lambda: False
                with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):
                    panoptoSync.main(sys.argv)
            finally:
                input_module.getpass = _og2
            _carica_elenco()
            print(f"Fatto. Trovate {len(ELENCO)} lezioni. Vai alla cella 5.")
        except Exception as e:
            msg = str(e).lower()
            if "depth limit" in msg or "authentication" in msg:
                print("Accesso non riuscito. Cause probabili:")
                print("  1) Username/password GIA sbagliati (cancella il file default.id in Drive e rifai).")
                print("  2) Login UniVR capriccioso: riesegui la cella.")
                print("  3) Troppi tentativi: aspetta qualche minuto.")
            else:
                print("Errore:", e)


## Cella 5 - Panopto: cerca la lezione

In [ ]:
#@title Scrivi qualche parola (nome corso o lezione) e premi play
CERCA = "" #@param {type:"string"}
#@markdown Esempi: `reti socket`, `ingegneria intro`, `markov`. Lascia vuoto per vedere le prime 40.

if not ELENCO:
    print("Prima esegui la cella 4 (accesso Panopto).")
else:
    toks = CERCA.lower().split()
    risultati = [t for t in ELENCO if all(tok in t[0].lower() for tok in toks)]
    print(f"{len(risultati)} risultati (mostro i primi 40):\n")
    import os as _os
    for i, (e, u, f) in enumerate(risultati[:40]):
        gia = "  (gia' scaricata)" if _os.path.exists(_os.path.join(LESSONS_DIR, f)) else ""
        print(f"[{i}]  {e}{gia}")
    RISULTATI = risultati
    print("\nAnnota il numero tra parentesi quadre della lezione che vuoi, poi vai alla cella 6.")


## Cella 6 - Panopto: scarica la lezione scelta

In [ ]:
#@title Metti il numero della lezione (dalla lista della cella 5) e premi play
NUMERO = 0 #@param {type:"integer"}

import os, sys
VIDEO_PATH = None
os.chdir("/content/panopto-sync")

if 'RISULTATI' not in globals() or not RISULTATI:
    print("Prima cerca una lezione nella cella 5.")
elif NUMERO < 0 or NUMERO >= len(RISULTATI):
    print(f"Numero non valido. Scegli tra 0 e {len(RISULTATI)-1}.")
else:
    etich, uuid, fname = RISULTATI[NUMERO]
    gia = os.path.join(LESSONS_DIR, fname)
    if os.path.exists(gia):
        VIDEO_PATH = gia
        print("Lezione gia' scaricata in memoria, la riuso (nessun nuovo download):")
        print(" ", VIDEO_PATH)
        print("Vai alla cella 9.")
    else:
        print("Scarico:", etich)
        print("Puo' richiedere qualche minuto in base alla durata...")
        try:
            import input as input_module
            with open("_scelta.txt", "w") as f:
                f.write(uuid + "\n")
            _og = input_module.getpass
            input_module.getpass = lambda p="": "colabkey"
            try:
                sys.argv = ["panoptoSync.py", "--idfile", ID_FILE, "single", "--output", LESSONS_DIR, "_scelta.txt"]
                import panoptoSync
                panoptoSync.main(sys.argv)
            finally:
                input_module.getpass = _og
            mkvs = [f for f in os.listdir(LESSONS_DIR) if f.lower().endswith((".mkv", ".mp4"))]
            VIDEO_PATH = max((os.path.join(LESSONS_DIR, f) for f in mkvs), key=os.path.getmtime)
            print("Scaricata:", VIDEO_PATH)
            print("Vai alla cella 9.")
        except Exception as e:
            print("Errore download:", e)


## Cella 7 - Video dal PC (esegui solo se NON usi Panopto)

In [ ]:
# Se hai gia' scaricato da Panopto, salta questa cella.
# Altrimenti esegui: si apre il selettore, scegli il video dal tuo computer.
import os, shutil
from google.colab import files
VIDEO_PATH = None
print("Si aprira' il selettore file. Scegli un video (mkv, mp4, avi, mov, webm)...")
up = files.upload()
name = list(up.keys())[0]
VIDEO_PATH = os.path.join("/content/lavoro", name)
shutil.move(name, VIDEO_PATH)
print("Caricato:", VIDEO_PATH)
print("Vai alla cella 9.")


## Cella 8 - Video da Google Drive (esegui solo se usi Drive)

In [ ]:
#@title Cerca un video nel tuo Drive e scegline uno
CERCA_DRIVE = "" #@param {type:"string"}
NUMERO_DRIVE = 0 #@param {type:"integer"}
#@markdown Prima esecuzione: lascia vuoto, esegui, guarda la lista.
#@markdown Poi scrivi una parola in CERCA_DRIVE e/o il NUMERO del file, riesegui.

import os, shutil
from google.colab import drive
VIDEO_PATH = None
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

VIDEO_EXTS = (".mp4", ".mkv", ".avi", ".mov", ".flv", ".webm")
trovati = []
for root, _, fs in os.walk("/content/drive/MyDrive"):
    for f in fs:
        if f.lower().endswith(VIDEO_EXTS):
            trovati.append(os.path.join(root, f))
trovati.sort()

q = CERCA_DRIVE.lower().strip()
filtrati = [p for p in trovati if q in p.lower()] if q else trovati
print(f"{len(filtrati)} video (mostro i primi 40):\n")
for i, p in enumerate(filtrati[:40]):
    print(f"[{i}]  {p}")

if filtrati and 0 <= NUMERO_DRIVE < len(filtrati):
    scelto = filtrati[NUMERO_DRIVE]
    VIDEO_PATH = os.path.join("/content/lavoro", os.path.basename(scelto))
    shutil.copy(scelto, VIDEO_PATH)
    print("\nScelto:", VIDEO_PATH, "- vai alla cella 9.")
else:
    print("\nScrivi il NUMERO del file voluto e riesegui.")


## Cella 9 - Cosa voglio ottenere?

In [ ]:
#@title Scegli tipo, dettaglio, lingua ed extra
TIPO = "Appunti" #@param ["Appunti", "Esercizi con soluzioni", "Foglio ripasso compatto"]
DETTAGLIO = "Equilibrato" #@param ["Sintetico", "Equilibrato", "Approfondito"]
LINGUA = "italiano" #@param {type:"string"}
Tabelle = True #@param {type:"boolean"}
Diagrammi = True #@param {type:"boolean"}
Domande_difficili = True #@param {type:"boolean"}
Immagini_slide = True #@param {type:"boolean"}
Formule_LaTeX = False #@param {type:"boolean"}

FORMATO = {"Appunti":"notes","Esercizi con soluzioni":"practice_problems","Foglio ripasso compatto":"formula_sheet"}[TIPO]
CONCISO = {"Sintetico":"short_hand","Equilibrato":"balanced","Approfondito":"deep_dive"}[DETTAGLIO]
STRUMENTI = []
if Tabelle: STRUMENTI.append("tables")
if Diagrammi: STRUMENTI.append("mermaid_diagrams")
if Domande_difficili: STRUMENTI.append("tricky_questions")
if Immagini_slide: STRUMENTI.append("screenshots")
if Formule_LaTeX: STRUMENTI.append("latex_support")

print("Scelte confermate:")
print(" tipo:", FORMATO, "| dettaglio:", CONCISO, "| lingua:", LINGUA)
print(" extra:", STRUMENTI)
print("Vai alla cella 10.")


## Cella 10 - Preparazione video

Se il video Panopto ha due tracce (schermo del prof + webcam), qui tiene solo la traccia con le slide (risoluzione maggiore), cosi' le immagini non escono nere o con l'icona della telecamera spenta.


In [ ]:
#@title Scelta traccia video
TRACCIA = "automatico" #@param ["automatico", "0", "1", "2"]
#@markdown `automatico` = tiene la traccia piu' grande (di solito le slide). Se le immagini escono
#@markdown sbagliate (es. la webcam del prof invece delle slide), rimetti la sorgente originale
#@markdown nelle celle 6/7/8 e qui scegli a mano `0`, `1` o `2` guardando la lista che compare.

import os, subprocess, json
assert 'VIDEO_PATH' in globals() and VIDEO_PATH and os.path.exists(VIDEO_PATH), "Nessun video: torna alle celle 6/7/8."

def _streams_video(path):
    o = subprocess.run(["ffprobe","-v","error","-select_streams","v",
                        "-show_entries","stream=index,width,height","-of","json",path],
                       capture_output=True, text=True)
    return json.loads(o.stdout).get("streams", [])

vs = _streams_video(VIDEO_PATH)

if len(vs) <= 1:
    print("Una sola traccia video, nessuna correzione necessaria. Vai alla cella 11.")
else:
    # anteprima: un fotogramma per ogni traccia (a 30 secondi, per evitare il nero iniziale)
    from IPython.display import Image, display
    print(f"Il video ha {len(vs)} tracce. Ecco un'anteprima di ciascuna:")
    for s in vs:
        prev = f"/content/_prev_{s['index']}.jpg"
        subprocess.run(["ffmpeg","-y","-loglevel","error","-ss","30","-i",VIDEO_PATH,
                        "-map", f"0:{s['index']}", "-frames:v","1", prev], check=True)
        print(f"\nTraccia {s['index']} ({s.get('width')}x{s.get('height')}):")
        display(Image(prev, width=380))

    if TRACCIA == "automatico":
        scelta = max(vs, key=lambda s: s.get("width",0)*s.get("height",0))
        print(f"\nScelta automatica: traccia {scelta['index']} (la piu' grande, di solito le slide).")
        print("Se dall'anteprima vedi che quella giusta e' un'altra, rimetti TRACCIA su 0/1/2 e riesegui.")
    else:
        cand = [s for s in vs if str(s["index"]) == TRACCIA]
        if not cand:
            raise ValueError(f"Traccia {TRACCIA} inesistente. Scegli tra: {[s['index'] for s in vs]}")
        scelta = cand[0]
        print(f"\nUso la traccia {scelta['index']} scelta da te.")

    fixed = os.path.splitext(VIDEO_PATH)[0] + "_scelta.mkv"
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",VIDEO_PATH,
                    "-map", f"0:{scelta['index']}", "-map","0:a","-c","copy", fixed], check=True)
    VIDEO_PATH = fixed
    print("Video pronto. Vai alla cella 11.")


## Cella 11 - Trascrizione dell'audio (Whisper)

Trasforma il parlato della lezione in testo. Se hai messo piu' chiavi Groq (separate da virgola nella cella 2) e una si esaurisce a meta', passa da sola alla successiva e riprende dal punto in cui era, senza ricominciare.


In [ ]:
#@title Qualita' della trascrizione
QUALITA = "Alta" #@param ["Alta", "Veloce"]
#@markdown Alta = whisper-large-v3 (piu' precisa). Veloce = whisper-large-v3-turbo (circa il doppio piu' rapida, qualita' di poco inferiore).

import math, subprocess, os, json
from groq import Groq

GROQ_KEYS = [k.strip() for k in GROQ_KEY.split(",") if k.strip()]
assert GROQ_KEYS, "Chiave Groq mancante (cella 2)."
MODEL = "whisper-large-v3" if QUALITA == "Alta" else "whisper-large-v3-turbo"

SRT_PATH = os.path.splitext(VIDEO_PATH)[0] + ".srt"
CHUNK_SEC = 600
PROG_DIR = os.path.splitext(VIDEO_PATH)[0] + "_trascr"
os.makedirs(PROG_DIR, exist_ok=True)

def _dur(p):
    o = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                        "-of","default=noprint_wrappers=1:nokey=1",p], capture_output=True, text=True, check=True)
    return float(o.stdout.strip())
def _ts(s):
    h=int(s//3600); m=int((s%3600)//60); ss=int(s%60); ms=int(round((s-int(s))*1000))
    return f"{h:02d}:{m:02d}:{ss:02d},{ms:03d}"

if os.path.exists(SRT_PATH):
    print("Trascrizione gia' presente, salto. Vai alla cella 12.")
else:
    dur = _dur(VIDEO_PATH)
    n = math.ceil(dur / CHUNK_SEC)
    # stima: Groq processa ~10 min di audio in ~15-30 s (Alta) o ~8-15 s (Veloce)
    per_blocco = 30 if QUALITA == "Alta" else 15
    stima_min = max(1, round(n * per_blocco / 60))
    print(f"Durata lezione: {dur/60:.0f} minuti, divisa in {n} blocco/i.")
    print(f"Tempo stimato di trascrizione: circa {stima_min}-{stima_min+2} minuti. Attendi...")
    print(f"Chiavi Groq disponibili: {len(GROQ_KEYS)}")

    ki = 0  # indice chiave corrente
    def _trascrivi(audio_path):
        global ki
        while ki < len(GROQ_KEYS):
            try:
                cli = Groq(api_key=GROQ_KEYS[ki])
                with open(audio_path, "rb") as f:
                    return cli.audio.transcriptions.create(
                        file=(os.path.basename(audio_path), f.read()),
                        model=MODEL, response_format="verbose_json")
            except Exception as e:
                print(f"    chiave {ki+1} non utilizzabile ({e}); passo alla successiva.")
                ki += 1
        raise RuntimeError("Tutte le chiavi Groq sono esaurite. Aggiungine altre nella cella 2 "
                           "e riesegui: riprendera' dal blocco dove si e' fermata.")

    for i in range(n):
        parziale = os.path.join(PROG_DIR, f"blocco_{i}.json")
        if os.path.exists(parziale):
            print(f"  blocco {i+1}/{n}: gia' fatto, salto.")
            continue
        ini = i * CHUNK_SEC
        ap = f"/content/a_{i}.mp3"
        subprocess.run(["ffmpeg","-y","-loglevel","error","-ss",str(ini),"-t",str(CHUNK_SEC),
                        "-i",VIDEO_PATH,"-vn","-ac","1","-ar","16000","-b:a","64k",ap], check=True)
        print(f"  blocco {i+1}/{n}: trascrivo...")
        r = _trascrivi(ap)
        blocco = []
        for s in r.segments:
            d = s if isinstance(s, dict) else s.__dict__
            blocco.append([float(d["start"])+ini, float(d["end"])+ini, str(d["text"]).strip()])
        json.dump(blocco, open(parziale, "w"))
        os.remove(ap)

    # unisce tutti i blocchi salvati nel file .srt finale
    segs = []
    for i in range(n):
        segs += json.load(open(os.path.join(PROG_DIR, f"blocco_{i}.json")))
    out = []; k = 0
    for a, b, t in segs:
        if not t: continue
        k += 1; out += [str(k), f"{_ts(a)} --> {_ts(b)}", t, ""]
    open(SRT_PATH, "w", encoding="utf-8").write("\n".join(out))
    print(f"Fatto: {k} sottotitoli. Vai alla cella 12.")


## Cella 12 - Generazione appunti e schemi

In [ ]:
import json, os
assert GEMINI_KEY, "Chiave Gemini mancante (cella 2)."

OUTPUT_DIR = f"/content/output_{FORMATO}"
CFG = "/content/config_lore.json"
PLIB = "/content/lore-engine/src/prompt_library.py"
MARK = "# --- lingua forzata ---"

src = open(PLIB, encoding="utf-8").read()
if MARK in src:
    src = src.split(MARK)[0].rstrip() + "\n"
if LINGUA:
    dire = (MARK + "\nPROMPT_LIBRARY['base']['instructions'] += "
            f"\"\\n\\nLANGUAGE: scrivi TUTTO l'output in {LINGUA}, inclusi titoli, tabelle, diagrammi e domande.\"\n")
    src = src.rstrip() + "\n\n" + dire
open(PLIB, "w", encoding="utf-8").write(src)

cfg = {"prompt_type":"captions","output_format":FORMATO,"conciseness":CONCISO,
       "tools":STRUMENTI,"model_name":"gemini-2.5-flash","lines_per_chunk":50}
json.dump(cfg, open(CFG,"w"), indent=2)
print("Generazione in corso, attendi...\n")

os.chdir("/content/lore-engine")
get_ipython().system(f'python src/main.py "{VIDEO_PATH}" --output "{OUTPUT_DIR}" --config "{CFG}" -y')

found = []
for r,_,fs in os.walk(OUTPUT_DIR):
    for f in fs:
        found.append(os.path.join(r,f))
if found:
    print("\nGenerati:")
    for f in found: print(" ", f)
    print("Vai alla cella 13.")
else:
    print("\nNessun file generato, controlla i messaggi sopra.")


## Cella 13 - Visualizza il risultato

In [ ]:
import re, os, glob, base64
import html as H
from IPython.display import HTML, display

md_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.md")))
assert md_files, "Nessun file da mostrare (controlla la cella 12)."

def _uri(p):
    return "data:image/jpeg;base64," + base64.b64encode(open(p,"rb").read()).decode()

def _render(mp):
    testo = open(mp, encoding="utf-8").read()
    merm = re.findall(r"```mermaid\n(.*?)\n```", testo, re.S)
    testo = re.sub(r"```mermaid\n.*?\n```", "%%M%%", testo, flags=re.S)
    def _img(m):
        s = m.group(2); p = s if os.path.isabs(s) else os.path.join(os.path.dirname(mp), s)
        return '<img src="'+_uri(p)+'">' if os.path.exists(p) else '<img src="'+H.escape(s)+'">'
    testo = re.sub(r"!\[([^\]]*)\]\(([^)]+)\)", _img, testo)
    out=[]; im=0; tab=False
    for riga in testo.splitlines():
        s = riga.strip()
        if s == "%%M%%":
            out.append('<div class="mermaid">'+merm[im]+"</div>"); im+=1; continue
        if s.startswith("|"):
            ce=[c.strip() for c in s.strip("|").split("|")]
            if not tab:
                out.append("<table><tr>"+"".join("<th>"+H.escape(c)+"</th>" for c in ce)+"</tr>"); tab=True
            elif set("".join(ce))<=set("-: "):
                pass
            else:
                out.append("<tr>"+"".join("<td>"+H.escape(c)+"</td>" for c in ce)+"</tr>")
            continue
        if tab: out.append("</table>"); tab=False
        if s.startswith("<img"): out.append(s)
        elif riga.startswith("### "): out.append("<h3>"+H.escape(riga[4:])+"</h3>")
        elif riga.startswith("## "): out.append("<h2>"+H.escape(riga[3:])+"</h2>")
        elif riga.startswith("# "): out.append("<h1>"+H.escape(riga[2:])+"</h1>")
        elif s=="---": out.append("<hr>")
        elif s=="": out.append("<br>")
        else:
            if s.startswith("*   "): cont,tg=s[4:],"li"
            elif s.startswith(("* ","- ")): cont,tg=s[2:],"li"
            else: cont,tg=riga,"p"
            t=H.escape(cont); t=re.sub(r"\*\*(.+?)\*\*",r"<b>\1</b>",t); t=re.sub(r"`(.+?)`",r"<code>\1</code>",t)
            out.append("<"+tg+">"+t+"</"+tg+">")
    if tab: out.append("</table>")
    stile=("body{font-family:system-ui,sans-serif;line-height:1.55;color:#222}"
           "li{margin-left:1.4em}code{background:#f0f0f0;padding:1px 5px;border-radius:3px}"
           "table{border-collapse:collapse;margin:1em 0;width:100%}td,th{border:1px solid #ccc;padding:6px 10px;text-align:left}th{background:#f5f5f5}"
           "img{max-width:100%;border:1px solid #ddd;border-radius:4px;margin:8px 0}.mermaid{text-align:center;margin:1.5em 0}")
    pagina=('<!doctype html><meta charset="utf-8">'
            '<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>'
            '<script>mermaid.initialize({startOnLoad:true});</script>'
            '<style>'+stile+'</style>'+"".join(out))
    open(mp[:-3]+".html","w",encoding="utf-8").write(pagina)
    return pagina

for mp in md_files:
    display(HTML(_render(mp)))
print("Fatto. Vai alla cella 14 per scaricare.")


## Cella 14 - Scarica il risultato

In [ ]:
#@title Dove salvare?
DESTINAZIONE = "Scarica sul PC" #@param ["Scarica sul PC", "Salva su Google Drive"]

import shutil, os
zip_path = shutil.make_archive(f"/content/schemi_{FORMATO}", "zip", OUTPUT_DIR)

if DESTINAZIONE == "Scarica sul PC":
    from google.colab import files
    print("Avvio download di", os.path.basename(zip_path))
    files.download(zip_path)
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/SchemiLezioni/{os.path.basename(OUTPUT_DIR)}"
    shutil.copytree(OUTPUT_DIR, dst, dirs_exist_ok=True)
    print("Salvato in Drive:", dst)
